# Qwen GPU Server for The Instructor

Run this notebook in Colab with a GPU runtime. It starts Qwen behind an OpenAI-compatible vLLM server, exposes it through a temporary Cloudflare Tunnel URL, and keeps the local RAG/Chroma setup unchanged.

Run the cells from top to bottom: install dependencies, start vLLM, start the Cloudflare tunnel, then smoke test the public endpoint.

In [3]:
# Qwen GPU Server for The Instructor
#
# Run this notebook in Colab with a GPU runtime. It starts a bigger Qwen model
# behind an OpenAI-compatible vLLM server, exposes it through a temporary
# Cloudflare Tunnel URL, and keeps your local RAG/Chroma setup unchanged.
#
# Recommended first model: Qwen/Qwen2.5-14B-Instruct-AWQ. It is a good quality
# jump over local 7B while still being realistic on Colab GPUs. If Colab gives
# you a small T4 and this fails for memory, switch to Qwen/Qwen2.5-7B-Instruct-AWQ.
#
# Run the cells from top to bottom: install, start vLLM, start tunnel, smoke test.

Run the cells from top to bottom: install, start vLLM, start tunnel, smoke test.

SyntaxError: invalid decimal literal (188848760.py, line 5)

In [4]:
# Install runtime dependencies. No ngrok token or account needed.
!pip3 install vllm -q
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

print("Installed vLLM and cloudflared.")

Installed vLLM and cloudflared.


In [5]:
# Start Qwen through vLLM in the background.
# If 14B fails due to GPU memory, change MODEL to "Qwen/Qwen2.5-7B-Instruct-AWQ" and rerun this cell.

import subprocess
import time
from pathlib import Path

import requests

MODEL = "Qwen/Qwen2.5-14B-Instruct-AWQ"
PORT = 8001
LOG_PATH = Path("/content/vllm.log")

cmd = [
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL,
    "--quantization", "awq",
    "--dtype", "half",
    "--max-model-len", "8192",
    "--gpu-memory-utilization", "0.90",
    "--host", "0.0.0.0",
    "--port", str(PORT),
]

print("Starting vLLM server...")
print(" ".join(cmd))
log_file = LOG_PATH.open("w")
vllm_process = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT)

print(f"Logs: {LOG_PATH}")
print("Waiting for /v1/models to become ready. First load can take several minutes...")

ready = False
for i in range(90):
    time.sleep(10)
    try:
        r = requests.get(f"http://127.0.0.1:{PORT}/v1/models", timeout=5)
        if r.ok:
            ready = True
            print("vLLM is ready.")
            print(r.json())
            break
    except Exception:
        pass
    print(f"Still loading... {10 * (i + 1)}s")

if not ready:
    print("vLLM did not become ready yet. Check the last 80 log lines below:")
    print("\n".join(LOG_PATH.read_text(errors="ignore").splitlines()[-80:]))

Starting vLLM server...
python -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-14B-Instruct-AWQ --quantization awq --dtype half --max-model-len 8192 --gpu-memory-utilization 0.90 --host 0.0.0.0 --port 8001
Logs: /content/vllm.log
Waiting for /v1/models to become ready. First load can take several minutes...
Still loading... 10s
Still loading... 20s
Still loading... 30s
Still loading... 40s
Still loading... 50s
Still loading... 60s
Still loading... 70s
Still loading... 80s
Still loading... 90s
Still loading... 100s
Still loading... 110s
Still loading... 120s
Still loading... 130s
Still loading... 140s
Still loading... 150s
Still loading... 160s
Still loading... 170s
Still loading... 180s
Still loading... 190s
Still loading... 200s
Still loading... 210s
Still loading... 220s
Still loading... 230s
Still loading... 240s
Still loading... 250s
Still loading... 260s
Still loading... 270s
Still loading... 280s
Still loading... 290s
Still loading... 300s
Still loading... 310s
Still l

: 

: 

In [6]:
# Start Cloudflare Tunnel. This prints a temporary public URL like:
# https://something-random.trycloudflare.com

import re
import subprocess
import time

import requests

# Confirm vLLM is still running before starting a tunnel.
try:
    models = requests.get(f"http://127.0.0.1:{PORT}/v1/models", timeout=10)
    print("Local vLLM status:", models.status_code, flush=True)
    print(models.text[:300], flush=True)
    models.raise_for_status()
except Exception as exc:
    raise RuntimeError("vLLM is not reachable on port 8001. Rerun the vLLM server cell first.") from exc

# Kill stale tunnels so cloudflared creates a fresh public URL.
subprocess.run(["pkill", "-f", "cloudflared"], check=False)
time.sleep(1)

cloudflared_process = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", f"http://localhost:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

CLOUDFLARE_URL = None
print("Starting Cloudflare Tunnel...", flush=True)

start = time.time()
while time.time() - start < 90:
    line = cloudflared_process.stdout.readline()
    if line:
        print(line, end="", flush=True)
        match = re.search(r"https://[-a-zA-Z0-9.]+\.trycloudflare\.com", line)
        if match:
            CLOUDFLARE_URL = match.group(0)
            break
    time.sleep(0.2)

if not CLOUDFLARE_URL:
    raise RuntimeError("Could not find Cloudflare Tunnel URL. If this repeats, restart the Colab runtime and run install -> vLLM -> tunnel.")

print("\nCOPY THIS URL for OPENAI_COMPAT_BASE_URL:", flush=True)
print(CLOUDFLARE_URL, flush=True)

Starting Cloudflare Tunnel...
2026-04-27T01:40:47Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-04-27T01:40:47Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-04-27T01:40:50Z INF +--------------------------------------------------------------------------------------------+
2026-04-27T01:40:50Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-04-27T01:40:50Z INF |  https://subsequent-temp

In [7]:
# Smoke test the public endpoint.

import requests

base_url = CLOUDFLARE_URL.rstrip("/")
print("Testing", base_url)

models = requests.get(f"{base_url}/v1/models", timeout=20)
print("/v1/models", models.status_code)
print(models.text[:500])

payload = {
    "model": MODEL,
    "messages": [
        {"role": "system", "content": "You are concise."},
        {"role": "user", "content": "Reply with exactly: Qwen GPU ready"},
    ],
    "temperature": 0,
    "max_tokens": 16,
}

chat = requests.post(f"{base_url}/v1/chat/completions", json=payload, timeout=120)
print("/v1/chat/completions", chat.status_code)
print(chat.json()["choices"][0]["message"]["content"])

Testing https://subsequent-template-pill-coupon.trycloudflare.com
/v1/models 200
{"object":"list","data":[{"id":"Qwen/Qwen2.5-14B-Instruct-AWQ","object":"model","created":1777254065,"owned_by":"vllm","root":"Qwen/Qwen2.5-14B-Instruct-AWQ","parent":null,"max_model_len":8192,"permission":[{"id":"modelperm-92e73b74a9397e50","object":"model_permission","created":1777254065,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}
/v1/chat/completions 200
Qwen GPU ready


: 

: 

: 

: 

## Connect Your Local App

Once the smoke test passes, copy the Cloudflare URL into your local backend command:

```bash
cd backend
LLM_PROVIDER=colab \
OPENAI_COMPAT_BASE_URL=https://YOUR-TRYCLOUDFLARE-URL \
OPENAI_COMPAT_MODEL=Qwen/Qwen2.5-14B-Instruct-AWQ \
OPENAI_COMPAT_MAX_TOKENS=1200 \
/Library/Frameworks/Python.framework/Versions/3.12/bin/python3.12 -m uvicorn main:app --reload
```

Then verify locally:

```bash
curl http://localhost:8000/llm/status
curl http://localhost:8000/rag/status
```

RAG stays local. The local backend retrieves examples from `data/chroma/`, then sends the final prompt to this Colab Qwen endpoint.